### 執行前準備

請先確認目前的 Jupyter kernel 使用 `iem_python` 環境，且已安裝 `pyautogui`、`pyscreeze` 與 `pillow`。若先前遇到 PyAutoGUI 無法載入 pyscreeze 的錯誤，請在該環境執行：

```bash
conda install -c conda-forge pyautogui pyscreeze pillow
```

偵測座標沿用此 notebook 先前成功取樣到仙人掌的螢幕位置：`x=167~181, y=410`；若瀏覽器視窗大小或縮放比例改變，請依遊戲畫面調整下方的座標常數。

In [3]:
import time
import pyautogui as pyg

# 仙人掌顏色是從原本成功偵測的畫面讀到的 RGB 值。
CACTUS_COLOR = (83, 83, 83)

# 在小恐龍前方、仙人掌底部高度橫向取樣。
# 這組座標依據 notebook 先前的成功偵測結果；若畫面配置不同，請調整這三個值。
DETECTION_X_START = 167
DETECTION_X_END = 181  # range 的結束值不包含在內
DETECTION_Y = 410

# 避免座標超出目前螢幕範圍，並在執行前顯示螢幕大小供校準。
screen_width, screen_height = pyg.size()
print(f"目前螢幕大小：{screen_width} x {screen_height}")
if not (0 <= DETECTION_X_START < DETECTION_X_END <= screen_width and 0 <= DETECTION_Y < screen_height):
    raise ValueError("偵測座標超出螢幕範圍，請依目前螢幕調整 DETECTION_X_START、DETECTION_X_END 和 DETECTION_Y。")

# 留 3 秒切換到 Chrome，並確保遊戲畫面取得鍵盤焦點。
print("3 秒後開始，請先切換到 Chrome 的離線小恐龍遊戲畫面！")
time.sleep(3)
pyg.press("space")  # 空白鍵開始遊戲
print("開始持續偵測仙人掌；在 Jupyter 按下停止即可結束。")

# 記錄上一輪是否已偵測到障礙物，避免同一根仙人掌仍在取樣區時重複起跳。
obstacle_was_detected = False

try:
    while True:
        # 逐點讀取小恐龍前方的螢幕像素；取樣到仙人掌顏色即視為有障礙物。
        obstacle_detected = any(
            pyg.pixel(x, DETECTION_Y) == CACTUS_COLOR
            for x in range(DETECTION_X_START, DETECTION_X_END)
        )

        # 只在障礙物剛出現時按一次空白鍵，避免連續多次跳躍。
        if obstacle_detected and not obstacle_was_detected:
            pyg.press("space")
        obstacle_was_detected = obstacle_detected

        # 稍作等待，降低不必要的高頻螢幕取樣與 CPU 使用率。
        time.sleep(0.01)
except KeyboardInterrupt:
    print("已停止小恐龍自動控制。")

目前螢幕大小：1920 x 1080
3 秒後開始，請先切換到 Chrome 的離線小恐龍遊戲畫面！
開始持續偵測仙人掌；在 Jupyter 按下停止即可結束。
已停止小恐龍自動控制。
